# Database Basics - Learning PostgreSQL with OLPA
## Learning Objectives
1. **Connect to PostgreSQL** using psycopg2 and SQLAlchemy
2. **Create tables** and understand schema design
3. **Load data** from CSV into database (Bronze layer)
4. **Query data** using SQL and pandas
5. **Understand database operations** (CRUD, indexing, transactions)

---

## Part 1: Understanding Database Setup

**Why use a database instead of CSV files?**
- ✅ **Performance**: Faster queries on large datasets with indexes
- ✅ **ACID compliance**: Data integrity with transactions
- ✅ **Concurrent access**: Multiple users/processes can access simultaneously
- ✅ **Data relationships**: Foreign keys maintain referential integrity
- ✅ **Production-ready**: Real-world data systems use databases

**Our Architecture**:
```
CSV Files → Bronze Layer → Silver Layer → Gold Layer
           (Raw)         (Cleaned)      (ML-Ready)
```

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import psycopg2
from psycopg2 import sql
from sqlalchemy import create_engine
import os
from datetime import datetime

print("✓ Libraries imported successfully")
print(f"Current directory: {os.getcwd()}")

## Part 2: Option A - Using Docker PostgreSQL (Recommended for Learning)

**Why Docker?**
- No installation needed
- Isolated environment
- Easy to reset and start fresh
- Same environment across machines

Run this in terminal:
```bash
# Start PostgreSQL in Docker
docker run -d \
  --name olpa-postgres \
  -e POSTGRES_USER=olpa_user \
  -e POSTGRES_PASSWORD=olpa_password \
  -e POSTGRES_DB=olpa_warehouse \
  -p 5432:5432 \
  postgres:15

# Check it's running
docker ps
```

**Or use the docker-compose.yml in the project root** (if it exists)

In [ ]:
# Database connection parameters
DB_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'database': 'olpa_warehouse',
    'user': 'olpa_user',
    'password': 'olpa_password'
}

print("Database configuration loaded")
print(f"Will connect to: {DB_CONFIG['user']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}")

## Part 3: Test Connection

**Two ways to connect:**
1. **psycopg2**: Low-level, more control, direct SQL
2. **SQLAlchemy**: High-level, ORM support, pandas integration

In [ ]:
# Method 1: psycopg2 (Low-level)
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cursor = conn.cursor()
    
    # Test query
    cursor.execute("SELECT version();")
    db_version = cursor.fetchone()
    
    print("✓ Connection successful (psycopg2)")
    print(f"PostgreSQL version: {db_version[0]}")
    
    cursor.close()
    conn.close()
    
except Exception as e:
    print(f"✗ Connection failed: {e}")
    print("\nTroubleshooting:")
    print("1. Is PostgreSQL running? (docker ps)")
    print("2. Are the credentials correct?")
    print("3. Is port 5432 available? (lsof -i :5432)")

In [ ]:
# Method 2: SQLAlchemy (High-level, better for pandas)
try:
    # Create connection string
    connection_string = f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
    
    # Create engine
    engine = create_engine(connection_string)
    
    # Test connection
    with engine.connect() as connection:
        result = connection.execute(sql.text("SELECT current_database();"))
        db_name = result.fetchone()[0]
    
    print("✓ Connection successful (SQLAlchemy)")
    print(f"Connected to database: {db_name}")
    
except Exception as e:
    print(f"✗ Connection failed: {e}")

## Part 4: Create a Simple Table (Learning SQL DDL)

**DDL (Data Definition Language)**:
- CREATE, ALTER, DROP
- Defines database structure

Let's create a simple test table first before using the full schema.

In [ ]:
# Create a simple test table
create_test_table = """
CREATE TABLE IF NOT EXISTS test_aircraft (
    id SERIAL PRIMARY KEY,
    aircraft_id VARCHAR(50) NOT NULL,
    model VARCHAR(50),
    manufacture_year INTEGER,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

try:
    conn = psycopg2.connect(**DB_CONFIG)
    cursor = conn.cursor()
    
    cursor.execute(create_test_table)
    conn.commit()
    
    print("✓ Table 'test_aircraft' created successfully")
    
    # Verify table exists
    cursor.execute("""
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema = 'public' AND table_name = 'test_aircraft';
    """)
    
    if cursor.fetchone():
        print("✓ Table verified in database")
    
    cursor.close()
    conn.close()
    
except Exception as e:
    print(f"✗ Error creating table: {e}")

## Part 5: Insert Data (Learning SQL DML)

**DML (Data Manipulation Language)**:
- INSERT, UPDATE, DELETE, SELECT
- Manipulates data in tables

In [ ]:
# Insert test data
test_data = [
    ('AC001', 'Boeing 737', 2015),
    ('AC002', 'Airbus A320', 2018),
    ('AC003', 'Boeing 777', 2012)
]

insert_query = """
INSERT INTO test_aircraft (aircraft_id, model, manufacture_year)
VALUES (%s, %s, %s)
ON CONFLICT DO NOTHING;
"""

try:
    conn = psycopg2.connect(**DB_CONFIG)
    cursor = conn.cursor()
    
    # Execute many (batch insert)
    cursor.executemany(insert_query, test_data)
    conn.commit()
    
    rows_inserted = cursor.rowcount
    print(f"✓ Inserted {rows_inserted} rows")
    
    cursor.close()
    conn.close()
    
except Exception as e:
    print(f"✗ Error inserting data: {e}")

## Part 6: Query Data (SELECT)

**Reading data from database**

In [ ]:
# Method 1: Using psycopg2
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cursor = conn.cursor()
    
    cursor.execute("SELECT * FROM test_aircraft;")
    rows = cursor.fetchall()
    
    print("Query results (psycopg2):")
    for row in rows:
        print(f"  {row}")
    
    cursor.close()
    conn.close()
    
except Exception as e:
    print(f"✗ Error querying data: {e}")

In [ ]:
# Method 2: Using pandas (MUCH easier!)
query = "SELECT * FROM test_aircraft;"

try:
    df = pd.read_sql_query(query, engine)
    print("✓ Query results as DataFrame:")
    display(df)
    
except Exception as e:
    print(f"✗ Error: {e}")

## Part 7: Load Real Data - Bronze Layer

Now let's load our actual OLPA sensor data!

In [ ]:
# Read the sensor data CSV
sensor_df = pd.read_csv('../data/raw/sensor_data.csv', parse_dates=['date'])

print(f"Loaded {len(sensor_df):,} sensor records")
print(f"\nColumns: {list(sensor_df.columns)}")
print(f"\nFirst few rows:")
display(sensor_df.head())

In [ ]:
# Create bronze_sensor_data table (simplified version)
create_bronze_sensor = """
CREATE TABLE IF NOT EXISTS bronze_sensor_data (
    id SERIAL PRIMARY KEY,
    aircraft_id VARCHAR(50) NOT NULL,
    cycle INTEGER,
    date DATE,
    temperature NUMERIC(8, 2),
    vibration NUMERIC(8, 4),
    pressure NUMERIC(8, 2),
    rpm NUMERIC(8, 2),
    altitude NUMERIC(10, 2),
    ambient_temp NUMERIC(8, 2),
    flight_hours NUMERIC(10, 2),
    will_fail_7days INTEGER,
    failed INTEGER,
    ingestion_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

-- Create indexes for better query performance
CREATE INDEX IF NOT EXISTS idx_bronze_aircraft_date 
    ON bronze_sensor_data(aircraft_id, date);
"""

try:
    conn = psycopg2.connect(**DB_CONFIG)
    cursor = conn.cursor()
    
    cursor.execute(create_bronze_sensor)
    conn.commit()
    
    print("✓ Table 'bronze_sensor_data' created successfully")
    print("✓ Index created for faster queries")
    
    cursor.close()
    conn.close()
    
except Exception as e:
    print(f"✗ Error: {e}")

In [ ]:
# Load data into database using pandas (EASIEST METHOD)
print(f"Loading {len(sensor_df):,} rows into database...")
print("This may take a minute...")

try:
    # Drop the 'id' column if it exists (database will auto-generate)
    sensor_df_to_load = sensor_df.copy()
    if 'id' in sensor_df_to_load.columns:
        sensor_df_to_load = sensor_df_to_load.drop(columns=['id'])
    
    # Load to database (if_exists='replace' will recreate table)
    # For incremental loads, use 'append'
    sensor_df_to_load.to_sql(
        name='bronze_sensor_data',
        con=engine,
        if_exists='append',  # 'replace' would drop and recreate
        index=False,
        method='multi',  # Faster batch insert
        chunksize=5000  # Insert in batches
    )
    
    print(f"✓ Data loaded successfully!")
    
except Exception as e:
    print(f"✗ Error loading data: {e}")

In [ ]:
# Verify the data was loaded
verify_query = """
SELECT 
    COUNT(*) as total_records,
    COUNT(DISTINCT aircraft_id) as unique_aircraft,
    MIN(date) as earliest_date,
    MAX(date) as latest_date,
    SUM(will_fail_7days) as failure_labels
FROM bronze_sensor_data;
"""

result_df = pd.read_sql_query(verify_query, engine)
print("\n✓ Database verification:")
display(result_df)

## Part 8: Practice SQL Queries

Now let's practice some common SQL patterns

In [ ]:
# Query 1: Get average sensor readings per aircraft
query1 = """
SELECT 
    aircraft_id,
    COUNT(*) as total_readings,
    ROUND(AVG(temperature), 2) as avg_temperature,
    ROUND(AVG(vibration), 4) as avg_vibration,
    ROUND(AVG(pressure), 2) as avg_pressure,
    SUM(will_fail_7days) as days_in_warning_window
FROM bronze_sensor_data
GROUP BY aircraft_id
ORDER BY days_in_warning_window DESC
LIMIT 10;
"""

result1 = pd.read_sql_query(query1, engine)
print("Top 10 aircraft by failure risk:")
display(result1)

In [ ]:
# Query 2: Time series for a specific aircraft
query2 = """
SELECT 
    date,
    cycle,
    temperature,
    vibration,
    will_fail_7days,
    failed
FROM bronze_sensor_data
WHERE aircraft_id = 'AC001'
ORDER BY date;
"""

result2 = pd.read_sql_query(query2, engine)
print(f"Time series for aircraft AC001: {len(result2)} days")
display(result2.head(10))

# Quick visualization
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(result2['cycle'], result2['temperature'], label='Temperature', color='red')
ax.set_xlabel('Cycle (Days)')
ax.set_ylabel('Temperature')
ax.set_title('Aircraft AC001 - Temperature Over Time')
ax.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Query 3: Window functions (advanced SQL)
query3 = """
SELECT 
    aircraft_id,
    date,
    temperature,
    -- Rolling average (last 7 days)
    AVG(temperature) OVER (
        PARTITION BY aircraft_id 
        ORDER BY date 
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) as temp_7day_avg,
    -- Rank by temperature
    RANK() OVER (PARTITION BY aircraft_id ORDER BY temperature DESC) as temp_rank
FROM bronze_sensor_data
WHERE aircraft_id IN ('AC001', 'AC002')
ORDER BY aircraft_id, date
LIMIT 20;
"""

result3 = pd.read_sql_query(query3, engine)
print("Window functions example (rolling average):")
display(result3)

## Part 9: Key Learnings Summary

**What you learned:**

1. ✅ **Database Connection**: Both psycopg2 (low-level) and SQLAlchemy (high-level)
2. ✅ **DDL Operations**: CREATE TABLE, CREATE INDEX
3. ✅ **DML Operations**: INSERT, SELECT
4. ✅ **Data Loading**: CSV → Database using pandas
5. ✅ **SQL Queries**: Aggregation, GROUP BY, Window Functions
6. ✅ **Best Practices**: Indexes, batch inserts, connection management

**Next Steps:**
- Notebook 03: Data Quality & Cleaning (Silver layer)
- Notebook 04: Feature Engineering (Gold layer)
- Notebook 05: Model Training

## Clean Up (Optional)

Drop test tables if you want to start fresh

In [ ]:
# Uncomment to drop tables
# drop_query = """
# DROP TABLE IF EXISTS test_aircraft CASCADE;
# DROP TABLE IF EXISTS bronze_sensor_data CASCADE;
# """

# conn = psycopg2.connect(**DB_CONFIG)
# cursor = conn.cursor()
# cursor.execute(drop_query)
# conn.commit()
# cursor.close()
# conn.close()
# print("✓ Tables dropped")